# Mini-projet Séries Temporelles — KO & PEP (2019-2024)
Exécuter toutes les cellules dans l'ordre. Prérequis : `pip install -r requirements.txt` (Python 3.10-3.12 conseillé pour TensorFlow). Graine fixée : 42.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../python'))   # pour importer common.py
from common import print_versions
print_versions()   # versions des packages documentées

# 01 — Données et analyse exploratoire (sections 1 et 2)
Source : Yahoo Finance, Coca-Cola (KO) et PepsiCo (PEP), journalier, 2019-2024.

In [ ]:
from common import *
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

print_versions()

## 1. Importation des données

In [ ]:
# Téléchargement une seule fois puis cache en CSV : le notebook reste reproductible même hors-ligne.
def telecharger():
    import yfinance as yf

    raw = yf.download(
        TICKERS,
        start=START,
        end=END_EXCL,
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=False,
        timeout=30
    )

    if raw.empty:
        raise RuntimeError(
            "Yahoo Finance n'a retourné aucune donnée. Réessayez."
        )

    for t in TICKERS:
        if t not in raw.columns.get_level_values(0):
            raise RuntimeError(f"Données absentes pour {t}")

        df = raw[t].dropna(how="all")

        if df.empty:
            raise RuntimeError(f"Aucune donnée valide pour {t}")

        df.to_csv(DATA_DIR / f"{t}.csv")
        print(f"{t}: {len(df)} lignes sauvegardées.")

if not all((DATA_DIR / f"{t}.csv").exists() for t in TICKERS):
    try:
        telecharger()
    except Exception as e:
        raise SystemExit(f"Téléchargement impossible ({e}). Téléchargez KO et PEP à la main sur "
                         "finance.yahoo.com (Historical Data, 2019-2024) et placez KO.csv / PEP.csv dans data/.")

In [ ]:
# Série temporelle native : DatetimeIndex trié, prix ajustés des dividendes (Adj Close).
px = load_raw("KO")  # aperçu du vecteur multidimensionnel complet
print(px.head(), "\n")
prices = load_prices()
F = make_features(prices)
n = len(F)
n_train = int(n * TRAIN_RATIO)          # split CHRONOLOGIQUE : jamais aléatoire sur une série temporelle
train, test = F.iloc[:n_train], F.iloc[n_train:]
print(f"{n} jours | train : {train.index[0].date()} -> {train.index[-1].date()} ({len(train)}) "
      f"| test : {test.index[0].date()} -> {test.index[-1].date()} ({len(test)})")

# Documentation de la source (exigée par la consigne 1.3)
(RES_DIR / "source.txt").write_text(
    "Source : Yahoo Finance (finance.yahoo.com) via yfinance, tickers KO et PEP, fréquence journalière.\n"
    f"Période : {prices.index[0].date()} -> {prices.index[-1].date()} ({n} jours de bourse).\n"
    "Variables brutes : Open, High, Low, Close, Adj Close, Volume.\n"
    "Transformation du fournisseur : 'Adj Close' est ajusté des dividendes et des splits ; 'Close' ne l'est que des splits.\n"
    "Série modélisée : log(Adj Close). Indicateurs dérivés : rendement log, moyenne mobile 20 j, volatilité réalisée 21 j.\n"
    f"Split chronologique : train 90 % ({len(train)} jours), test 10 % ({len(test)} jours).\n", encoding="utf-8")
pd.concat({t: load_raw(t).describe().T for t in TICKERS}).to_csv(RES_DIR / "describe_raw.csv")

## 2. Analyse exploratoire

In [ ]:
# Prix bruts (échelles différentes) puis rebasés base 100 pour comparer les dynamiques.
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
prices.plot(ax=ax[0]); ax[0].set_title("Prix ajustés (USD) — échelles différentes")
(prices / prices.iloc[0] * 100).plot(ax=ax[1]); ax[1].set_title("Prix rebasés (base 100 au 01/01/2019)")
for a in ax:
    a.axvline(test.index[0], color="k", ls="--", lw=1)  # frontière train / test
ax[1].text(test.index[0], ax[1].get_ylim()[1] * 0.95, " test", va="top")
savefig(fig, "fig01_prix.png")

# Indicateurs dérivés : moyenne mobile, rendements, volatilité réalisée.
fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax[0].plot(prices["KO"], lw=0.8, label="KO"); ax[0].plot(F["ma20_KO"], label="MM 20 j"); ax[0].legend(); ax[0].set_title("KO et moyenne mobile 20 j")
F[["ret_KO", "ret_PEP"]].plot(ax=ax[1], lw=0.6); ax[1].set_title("Rendements logarithmiques")
F[["vol21_KO", "vol21_PEP"]].plot(ax=ax[2]); ax[2].set_title("Volatilité réalisée 21 j (annualisée) — clusters de volatilité")
savefig(fig, "fig02_indicateurs.png")

# Statistiques descriptives sur le TRAIN (utilisées aussi pour le prompt LLM, section 5).
desc = pd.DataFrame({c: {"moyenne": train[c].mean(), "variance": train[c].var(), "ecart_type": train[c].std(),
                         "skewness": train[c].skew(), "kurtosis_excess": train[c].kurt(),
                         "min": train[c].min(), "max": train[c].max(),
                         "JB_p": stats.jarque_bera(train[c].dropna())[1]} for c in ["ret_KO", "ret_PEP"]})
desc.to_csv(RES_DIR / "stats_descriptives.csv")
print("\nStatistiques descriptives des rendements (train) :\n", desc.round(6))
print("Corrélation des rendements KO/PEP :", round(train["ret_KO"].corr(train["ret_PEP"]), 3))

In [ ]:
# ACF/PACF sur la série brute puis différenciée : l'ACF lente du niveau suggère une racine unitaire.
y = train["log_KO"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(y, lags=40, ax=ax[0], title="ACF log(KO) — série brute"); plot_pacf(y, lags=40, ax=ax[1], title="PACF log(KO) — série brute")
savefig(fig, "fig03_acf_pacf_niveau.png")
dy = y.diff().dropna()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(dy, lags=40, ax=ax[0], title="ACF Δlog(KO) — série différenciée"); plot_pacf(dy, lags=40, ax=ax[1], title="PACF Δlog(KO) — série différenciée")
savefig(fig, "fig04_acf_pacf_diff.png")

In [ ]:
# Décomposition STL robuste, période 252 (≈ 1 an de bourse) : sépare tendance, saisonnalité annuelle, reste.
stl = STL(pd.Series(y.values), period=252, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(11, 8); savefig(fig, "fig05_stl_KO.png")
force_tendance = max(0, 1 - np.var(stl.resid) / np.var(stl.trend + stl.resid))
force_saison = max(0, 1 - np.var(stl.resid) / np.var(stl.seasonal + stl.resid))
pd.Series({"force_tendance": force_tendance, "force_saisonnalite": force_saison}).to_csv(RES_DIR / "stl_forces.csv", header=["valeur"])
print(f"\nSTL : force de la tendance = {force_tendance:.3f} ; force de la saisonnalité = {force_saison:.3f} (0 = nulle, 1 = très forte)")

In [ ]:
# Valeurs aberrantes : règle IQR (1.5) calibrée sur le train, appliquée à tout l'échantillon.
def iqr_flags(x, ref, k=1.5):
    q1, q3 = ref.quantile([0.25, 0.75]); i = q3 - q1
    return (x < q1 - k * i) | (x > q3 + k * i)

flags = pd.DataFrame({f"flag_{t}": iqr_flags(F[f"ret_{t}"], train[f"ret_{t}"]) for t in TICKERS})
flags.loc[flags.index[0]] = False  # le 1er rendement est NaN
out = pd.concat([F[["ret_KO", "ret_PEP"]], flags], axis=1)
out[out[["flag_KO", "flag_PEP"]].any(axis=1)].to_csv(RES_DIR / "outliers.csv")
print("\nValeurs aberrantes (IQR) :", flags.sum().to_dict())
print("5 plus gros |rendements| KO :\n", F["ret_KO"].abs().nlargest(5).round(4))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].boxplot([train["ret_KO"].dropna(), train["ret_PEP"].dropna()]); ax[0].set_xticklabels(TICKERS); ax[0].set_title("Boxplot des rendements (train)")
ax[1].plot(F["ret_KO"], lw=0.5); ax[1].scatter(F.index[flags["flag_KO"]], F["ret_KO"][flags["flag_KO"]], c="r", s=8, label="aberrant (IQR)")
ax[1].legend(); ax[1].set_title("Rendements KO et points aberrants")
savefig(fig, "fig06_outliers.png")
# Traitement : CONSERVATION + variable indicatrice (flags). Ce sont de vrais chocs de marché (ex. mars 2020),
# les supprimer fausserait le risque ; GARCH est précisément fait pour les absorber.

In [ ]:
# Stationnarité : ADF (H0 = racine unitaire) ET KPSS (H0 = stationnaire) -> hypothèses nulles opposées.
rows = []
for name, s in [("log_KO", train["log_KO"]), ("log_PEP", train["log_PEP"]), ("ret_KO", train["ret_KO"]), ("ret_PEP", train["ret_PEP"])]:
    s = s.dropna()
    adf_p = adfuller(s, regression="c", autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    if adf_p < 0.05 and kpss_p >= 0.05:
        concl = "stationnaire"
    elif adf_p >= 0.05 and kpss_p < 0.05:
        concl = "NON stationnaire (racine unitaire)"
    else:
        concl = "ambigu (les deux tests divergent)"
    rows.append({"serie": name, "ADF_p": adf_p, "KPSS_p": kpss_p, "conclusion": concl})
stat = pd.DataFrame(rows)
stat.to_csv(RES_DIR / "stationnarite.csv", index=False)
print("\n", stat.round(4).to_string(index=False))
d_ordre = 1 if (stat.loc[stat.serie == "log_KO", "conclusion"].iloc[0].startswith("NON")
                and stat.loc[stat.serie == "ret_KO", "conclusion"].iloc[0] == "stationnaire") else 0
print(f"\n>>> Conclusion : ordre de différenciation d = {d_ordre} pour log(KO) "
      "(niveau non stationnaire, rendement stationnaire => d=1)." if d_ordre else
      "\n>>> Les tests ne concluent pas proprement à d=1 : à discuter dans le rapport (voir tableau).")
log_json("key_results.json", d_ordre=d_ordre, n_train=n_train, n_test=len(test),
         corr_rendements=float(train["ret_KO"].corr(train["ret_PEP"])), force_tendance=float(force_tendance),
         force_saisonnalite=float(force_saison), n_aberrants_KO=int(flags["flag_KO"].sum()))

# 02 — ARIMA / SARIMA avec diagnostic complet (section 3.1) + prévisions glissantes h=10
Série modélisée : log(Adj Close) de KO.

In [ ]:
from common import *
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf

F = make_features(load_prices())
n_train, origs = origins(len(F))
logKO = F["log_KO"].values
y_tr = logKO[:n_train]
d = read_json("key_results.json").get("d_ordre", 1) or 1   # d issu des tests ADF/KPSS (script 01)

## Identification à partir de l'ACF/PACF (approche manuelle)

In [ ]:
# Règle de lecture : p = dernier retard significatif de la PACF, q = dernier retard significatif de l'ACF
# (seuil ±1.96/√n, 5 premiers retards). C'est un POINT DE DÉPART à valider sur fig04 ; on peut le forcer :
ORDRE_FORCE = None   # ex. (1, 1, 1) pour imposer un ordre après lecture des graphiques
dy = np.diff(y_tr, n=d)
seuil = 1.96 / np.sqrt(len(dy))
a, pc = acf(dy, nlags=5, fft=False)[1:], pacf(dy, nlags=5)[1:]
p_man = max([k + 1 for k, v in enumerate(pc) if abs(v) > seuil], default=0)
q_man = max([k + 1 for k, v in enumerate(a) if abs(v) > seuil], default=0)
order_man = ORDRE_FORCE or (p_man, d, q_man)
print(f"Seuil de significativité ±{seuil:.3f} | ACF(1..5)={np.round(a, 3)} | PACF(1..5)={np.round(pc, 3)}")
print("Ordre manuel (ACF/PACF) :", order_man)

## Recherche automatique (pmdarima, sinon grille AIC de secours) et SARIMA

In [ ]:
def grille_aic(y, d, max_p=3, max_q=3):
    """Secours si pmdarima n'est pas disponible : on minimise l'AIC sur une petite grille (p,d,q)."""
    best = None
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                r = ARIMA(y, order=(p, d, q)).fit()
            except Exception:
                continue
            if best is None or r.aic < best[0]:
                best = (r.aic, (p, d, q))
    return best[1]

try:
    import pmdarima as pm
    order_auto = pm.auto_arima(y_tr, d=d, seasonal=False, information_criterion="aic", stepwise=True,
                               suppress_warnings=True, error_action="ignore").order
    sar = pm.auto_arima(y_tr, d=d, seasonal=True, m=5, D=0, information_criterion="aic", stepwise=True,
                        suppress_warnings=True, error_action="ignore")
    order_sar, seas_sar = sar.order, sar.seasonal_order
    print("auto_arima (pmdarima) utilisé")
except Exception as e:
    print(f"pmdarima indisponible ({type(e).__name__}) -> grille AIC de secours")
    order_auto = grille_aic(y_tr, d)
    cands = [(1, 0, 0, 5), (0, 0, 1, 5), (1, 0, 1, 5)]   # saisonnalité hebdomadaire m=5 jours de bourse
    order_sar = order_auto
    seas_sar = min(cands, key=lambda s: ARIMA(y_tr, order=order_auto, seasonal_order=s).fit().aic)
print("Ordre auto :", order_auto, "| SARIMA :", order_sar, seas_sar)
# m=5 car en données boursières journalières le seul cycle court plausible est hebdomadaire ;
# la saisonnalité annuelle est testée par la STL (script 01).

NULL = (0, 0, 0, 0)
MODELES = {"ARIMA_manuel": (order_man, NULL), "ARIMA_auto": (order_auto, NULL), "SARIMA": (order_sar, seas_sar)}
print("Comparaison manuel vs auto :", "IDENTIQUES" if order_man == order_auto else "DIFFÉRENTS (à commenter dans le rapport)")

## Estimation et diagnostic complet des résidus

In [ ]:
rows, resids = [], {}
for name, (order, seas) in MODELES.items():
    with Timer(name):
        res = ARIMA(y_tr, order=order, seasonal_order=seas).fit()
    skip = order[1] + seas[1] * seas[3]            # on retire les 1ers résidus (initialisation de la différenciation)
    r = res.resid[max(skip, 1):]
    k = order[0] + order[2] + seas[0] + seas[2]    # nb de paramètres ARMA : corrige les degrés de liberté de Ljung-Box
    dg = diag_tests(r, model_df=k)
    print_diag(f"{name} {order}{seas if seas != NULL else ''}", dg)
    plot_diag(r, f"Diagnostic des résidus — {name}", f"fig0{7 + list(MODELES).index(name)}_resid_{name}.png")
    rows.append({"modele": name, "ordre": str(order), "saisonnier": str(seas), "AIC": res.aic, "BIC": res.bic, **dg})
    resids[name] = (r, order, seas)
cmp_ = pd.DataFrame(rows)
cmp_.to_csv(RES_DIR / "diag_arima.csv", index=False)
print("\n", cmp_[["modele", "ordre", "saisonnier", "AIC", "BIC", "LB10_p", "JB_p", "ARCH10_p"]].round(4).to_string(index=False))

# Meilleur modèle (AIC) : ses résidus servent au test ARCH-LM de la section 3.3.
best = cmp_.loc[cmp_["AIC"].idxmin(), "modele"]
log_json("key_results.json", best_arima=best, best_arima_order=str(MODELES[best][0]), best_arima_seasonal=str(MODELES[best][1]),
         arima_manual_order=str(order_man), arima_auto_order=str(order_auto),
         arima_arch_p=float(cmp_.loc[cmp_.modele == best, "ARCH10_p"].iloc[0]),
         arima_lb10_p=float(cmp_.loc[cmp_.modele == best, "LB10_p"].iloc[0]))
pd.Series(resids[best][0]).to_csv(RES_DIR / "resid_best_arima.csv", index=False, header=["resid"])
print("Meilleur ARIMA/SARIMA (AIC) :", best)

## Prévision h=10 avec intervalle de confiance (premier bloc du test)

In [ ]:
order_b, seas_b = MODELES[best]
res_b = ARIMA(y_tr, order=order_b, seasonal_order=seas_b).fit()
fc = res_b.get_forecast(H)
mean, ci = fc.predicted_mean, fc.conf_int(alpha=0.05)
dates = F.index
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(dates[n_train - 60:n_train], np.exp(y_tr[-60:]), label="train (60 derniers jours)")
ax.plot(dates[n_train:n_train + H], np.exp(logKO[n_train:n_train + H]), "k", label="observé (test)")
ax.plot(dates[n_train:n_train + H], np.exp(mean), "r--", label=f"prévision {best}")
ax.fill_between(dates[n_train:n_train + H], np.exp(ci[:, 0]), np.exp(ci[:, 1]), color="r", alpha=0.2, label="IC 95 %")
ax.legend(); ax.set_title(f"Prévision à {H} jours — {best} (KO, USD)")
savefig(fig, "fig10_prevision_arima.png")

## Prévisions glissantes (walk-forward) pour la comparaison finale (section 6)
Pour chaque bloc de 10 jours du test : on réestime sur tout l'historique connu (ordre fixé), on prévoit 10 pas.

In [ ]:
def rolling(fit_forecast):
    return np.array([np.exp(fit_forecast(logKO[:o])) for o in origs])

save_forecasts("naive", rolling(lambda h: np.repeat(h[-1], H)))   # baseline marche aléatoire : dernier prix connu
for name, (order, seas) in MODELES.items():
    save_forecasts(name, rolling(lambda h, o=order, s=seas: ARIMA(h, order=o, seasonal_order=s).fit().forecast(H)))
    print("  prévisions glissantes OK :", name)

# 03 — VAR, causalité de Granger, cointégration et VECM (section 3.2)
Variables : KO et PEP. VAR sur les rendements (stationnaires), cointégration sur les log-prix (I(1)).

In [ ]:
from common import *
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
from statsmodels.tsa.vector_ar.vecm import select_order as vecm_select_order

F = make_features(load_prices())
n_train, origs = origins(len(F))
logKO = F["log_KO"].values
ret = F[["ret_KO", "ret_PEP"]]
ret_tr = ret.iloc[1:n_train].reset_index(drop=True)          # rendements stationnaires (d=1 sur les log-prix)

## VAR sur les rendements

In [ ]:
# Ordre choisi par BIC (plus parcimonieux que l'AIC sur ~1300 observations) ; le tableau montre aussi l'AIC.
sel = VAR(ret_tr).select_order(10)
print(sel.summary())
p_var = max(1, sel.selected_orders["bic"])
with Timer("VAR"):
    var_res = VAR(ret_tr).fit(p_var)
print(f"\nVAR({p_var}) estimé | stable : {var_res.is_stable()} (racines hors du cercle unité = OK)")

# Causalité de Granger dans les deux sens (test F). H0 : la variable 'causante' n'aide pas à prévoir l'autre.
g1 = var_res.test_causality("ret_KO", ["ret_PEP"], kind="f")
g2 = var_res.test_causality("ret_PEP", ["ret_KO"], kind="f")
print("\nGranger PEP -> KO :", verdict(g1.pvalue, "PEP ne cause pas KO au sens de Granger"))
print("Granger KO -> PEP :", verdict(g2.pvalue, "KO ne cause pas PEP au sens de Granger"))
print("(Granger = pouvoir prédictif, pas une causalité économique.)")

# Diagnostic des résidus du VAR : portemanteau multivarié, normalité, puis tests par équation.
white, norm = var_res.test_whiteness(nlags=20), var_res.test_normality()
print("Portemanteau (autocorrélation des résidus) :", verdict(white.pvalue, "résidus = bruit blanc"))
print("Normalité (Jarque-Bera multivarié)         :", verdict(norm.pvalue, "résidus normaux"))
diag_rows = []
for col in ["ret_KO", "ret_PEP"]:
    dg = diag_tests(var_res.resid[col].values, model_df=p_var)
    print_diag(f"VAR équation {col}", dg)
    diag_rows.append({"equation": col, **dg})
pd.DataFrame(diag_rows).to_csv(RES_DIR / "diag_var.csv", index=False)
plot_diag(var_res.resid["ret_KO"].values, f"Diagnostic des résidus — VAR({p_var}), équation KO", "fig11_resid_VAR_KO.png")

## Cointégration sur les log-prix

In [ ]:
lv = F[["log_KO", "log_PEP"]].reset_index(drop=True)
lv_tr = lv.iloc[:n_train]
# Engle-Granger (2 séries) : H0 = PAS de cointégration.
eg_t, eg_p, _ = coint(lv_tr["log_KO"], lv_tr["log_PEP"], trend="c", autolag="aic")
print("Engle-Granger :", verdict(eg_p, "pas de cointégration"))
# Johansen (trace) : rang de cointégration r. Le nombre de retards vient du critère BIC sur les niveaux.
k_diff = max(1, vecm_select_order(lv_tr, maxlags=10, deterministic="co").bic)
rank = select_coint_rank(lv_tr, det_order=0, k_ar_diff=k_diff, method="trace", signif=0.05)
print(f"Johansen (trace, 5 %) : rang de cointégration r = {rank.rank} (k_ar_diff = {k_diff})")
print(rank.summary())

# Spread de long terme KO - b·PEP (régression des niveaux) : doit osciller autour d'une moyenne si cointégration.
b, c = np.polyfit(lv_tr["log_PEP"], lv_tr["log_KO"], 1)
spread = lv["log_KO"] - b * lv["log_PEP"] - c
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(F.index, spread, lw=0.8); ax.axhline(0, color="k"); ax.axvline(F.index[n_train], color="r", ls="--")
ax.set_title(f"Écart de long terme  log(KO) − {b:.2f}·log(PEP) − {c:.2f}  (pointillé rouge = début du test)")
savefig(fig, "fig12_spread_cointegration.png")
log_json("key_results.json", var_order=int(p_var), granger_PEP_to_KO_p=float(g1.pvalue), granger_KO_to_PEP_p=float(g2.pvalue),
         var_whiteness_p=float(white.pvalue), var_stable=bool(var_res.is_stable()),
         engle_granger_p=float(eg_p), johansen_rank=int(rank.rank), vecm_lags_diff=int(k_diff),
         coint_beta_PEP=float(b))

## VECM (seulement si cointégration détectée)

In [ ]:
vecm_ok = rank.rank >= 1
if vecm_ok:
    with Timer("VECM"):
        vecm = VECM(lv_tr, k_ar_diff=k_diff, coint_rank=1, deterministic="co").fit()
    alpha, beta, pa = vecm.alpha[:, 0], vecm.beta[:, 0], vecm.pvalues_alpha[:, 0]
    print("\nVECM : relation de long terme normalisée sur la 1re variable (log_KO) :")
    print(f"  beta (log_KO, log_PEP) = {np.round(beta, 4)}")
    print(f"  alpha (vitesse d'ajustement) : KO = {alpha[0]:.4f} (p={pa[0]:.3f}) | PEP = {alpha[1]:.4f} (p={pa[1]:.3f})")
    for nm, al, pv in [("KO", alpha[0], pa[0]), ("PEP", alpha[1], pa[1])]:
        if al < 0 and pv < 0.05:
            print(f"  -> {nm} corrige l'écart : environ {abs(al)*100:.1f} % de l'écart disparaît chaque jour, "
                  f"demi-vie ≈ {np.log(0.5)/np.log(1+al):.0f} jours de bourse.")
        else:
            print(f"  -> {nm} ne corrige pas significativement l'écart (alpha non significatif ou de signe non correcteur).")
    log_json("key_results.json", vecm_alpha_KO=float(alpha[0]), vecm_alpha_PEP=float(alpha[1]),
             vecm_alpha_KO_p=float(pa[0]), vecm_alpha_PEP_p=float(pa[1]))
else:
    print("\nPas de cointégration détectée -> pas de VECM (un VAR en différences suffit). À expliquer dans le rapport.")

## Prévisions glissantes h=10 (VAR et VECM) pour la comparaison finale

In [ ]:
def var_forecast(o):
    r = ret.iloc[1:o].reset_index(drop=True)
    rf = VAR(r).fit(p_var).forecast(r.values[-p_var:], steps=H)[:, 0]      # rendements prévus de KO
    return np.exp(logKO[o - 1] + np.cumsum(rf))                              # on recompose le prix

save_forecasts("VAR", np.array([var_forecast(o) for o in origs]))
if vecm_ok:
    save_forecasts("VECM", np.array([
        np.exp(VECM(lv.iloc[:o], k_ar_diff=k_diff, coint_rank=1, deterministic="co").fit().predict(steps=H)[:, 0])
        for o in origs]))
print("Prévisions glissantes OK : VAR" + (", VECM" if vecm_ok else ""))

# 04 — Test ARCH-LM et modèles GARCH (section 3.3)
Série : rendements journaliers de KO (en %), échantillon d'apprentissage.

In [ ]:
from common import *
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

F = make_features(load_prices())
n_train, _ = origins(len(F))
r = (F["ret_KO"].iloc[1:n_train] * 100)          # en % : l'optimiseur de GARCH converge mieux à cette échelle

In [ ]:
# 1) Faut-il un GARCH ? Test ARCH-LM sur les résidus du meilleur ARIMA (script 02) et sur les rendements.
res_arima = pd.read_csv(RES_DIR / "resid_best_arima.csv")["resid"].values
p_arch_arima = het_arch(res_arima, nlags=10)[1]
p_arch_ret = het_arch(r.values - r.mean(), nlags=10)[1]
print("ARCH-LM résidus du meilleur ARIMA :", verdict(p_arch_arima, "variance conditionnelle constante"))
print("ARCH-LM rendements KO             :", verdict(p_arch_ret, "variance conditionnelle constante"))
print("=> si H0 rejetée : la variance varie dans le temps, un GARCH est justifié." )

In [ ]:
# 2) Sélection : GARCH(1,1) de référence comparé à des ordres supérieurs, lois normale et Student-t (BIC).
rows, fits = [], {}
for (p, q) in [(1, 1), (1, 2), (2, 1), (2, 2)]:
    for dist in ["normal", "t"]:
        f = arch_model(r, mean="Constant", vol="GARCH", p=p, q=q, dist=dist).fit(disp="off")
        fits[(p, q, dist)] = f
        rows.append({"modele": f"GARCH({p},{q})", "loi": dist, "logL": f.loglikelihood, "AIC": f.aic, "BIC": f.bic})
sel = pd.DataFrame(rows)
sel.to_csv(RES_DIR / "garch_selection.csv", index=False)
print("\n", sel.round(2).to_string(index=False))
best_row = sel.loc[sel["BIC"].idxmin()]
key_best = (int(best_row.modele[6]), int(best_row.modele[8]), best_row["loi"])
print("\nMeilleur selon BIC :", best_row.modele, best_row["loi"])

In [ ]:
# 3) Diagnostic des résidus STANDARDISÉS : doivent redevenir sans autocorrélation et homoscédastiques.
def diag_garch(f, label):
    z = np.asarray(f.std_resid, dtype=float); z = z[~np.isnan(z)]
    lb_z = acorr_ljungbox(z, lags=[10, 20], return_df=True)["lb_pvalue"]
    lb_z2 = acorr_ljungbox(z ** 2, lags=[10, 20], return_df=True)["lb_pvalue"]   # z² : variance résiduelle ?
    d = {"modele": label, "LB10_z_p": lb_z.iloc[0], "LB20_z_p": lb_z.iloc[1], "LB10_z2_p": lb_z2.iloc[0],
         "LB20_z2_p": lb_z2.iloc[1], "ARCH10_p": het_arch(z, nlags=10)[1], "JB_p": stats.jarque_bera(z)[1]}
    print(f"\n--- Diagnostic résidus standardisés : {label}")
    print("  Ljung-Box z   :", verdict(d["LB10_z_p"], "pas d'autocorrélation"))
    print("  Ljung-Box z²  :", verdict(d["LB10_z2_p"], "pas d'autocorrélation de la variance"))
    print("  ARCH-LM z     :", verdict(d["ARCH10_p"], "homoscédasticité retrouvée"))
    print("  Jarque-Bera z :", verdict(d["JB_p"], "résidus standardisés normaux"))
    return d

f11 = fits[(1, 1, "normal")]
f_best = fits[key_best]
dg = pd.DataFrame([diag_garch(f11, "GARCH(1,1) normal"), diag_garch(f_best, f"{best_row.modele} {best_row['loi']} (meilleur BIC)")])
dg.to_csv(RES_DIR / "diag_garch.csv", index=False)

In [ ]:
# 4) Persistance et demi-vie des chocs de volatilité (interprétation économique du modèle retenu).
par = f_best.params
persist = sum(v for k, v in par.items() if k.startswith(("alpha", "beta")))
half = np.log(0.5) / np.log(persist) if 0 < persist < 1 else np.inf
print(f"\nPersistance (Σα+Σβ) = {persist:.4f} -> demi-vie d'un choc de volatilité ≈ {half:.1f} jours")
vol_prev = np.sqrt(f_best.forecast(horizon=H).variance.iloc[-1].values)
pd.DataFrame({"horizon": range(1, H + 1), "vol_journaliere_%": vol_prev}).to_csv(RES_DIR / "garch_vol_forecast.csv", index=False)
log_json("key_results.json", garch_best=f"{best_row.modele} {best_row['loi']}", garch_persistence=float(persist),
         garch_half_life_days=float(half), arch_lm_arima_p=float(p_arch_arima), arch_lm_returns_p=float(p_arch_ret),
         garch_z2_lb10_p=float(dg["LB10_z2_p"].iloc[1]), garch_arch_z_p=float(dg["ARCH10_p"].iloc[1]))

In [ ]:
# 5) Volatilité conditionnelle dans le temps, reliée à des événements connus (à vérifier sur le graphique).
vol = pd.Series(np.asarray(f_best.conditional_volatility), index=F.index[1:n_train])
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(r.index if hasattr(r, "index") else vol.index, r.values, lw=0.5); ax[0].set_title("Rendements KO (%)")
ax[1].plot(vol, lw=0.9); ax[1].set_title(f"Volatilité conditionnelle estimée — {best_row.modele} {best_row['loi']} (écart-type journalier, %)")
for date, txt in [("2020-03-16", "Krach COVID-19"), ("2022-03-16", "1re hausse des taux Fed")]:
    for a in ax:
        a.axvline(pd.Timestamp(date), color="r", ls="--", lw=0.8)
    ax[1].text(pd.Timestamp(date), vol.max() * 0.9, " " + txt, color="r", rotation=90, va="top")
savefig(fig, "fig13_garch_volatilite.png")

z = np.asarray(f_best.std_resid, dtype=float); z = z[~np.isnan(z)]
plot_diag(z, f"Résidus standardisés — {best_row.modele} {best_row['loi']}", "fig14_garch_residus.png")

# 05 — LSTM et GRU avec validation croisée temporelle (section 4.1)
Implémenté en Python/Keras uniquement (socle avancé : un seul langage autorisé).
Entrée : fenêtre glissante de L jours de [rendement KO, rendement PEP, volatilité 21 j KO].
Sortie : les H=10 prochains rendements de KO (prévision directe multi-pas), recomposés en prix.

In [ ]:
from common import *
import tensorflow as tf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)   # graine fixée pour numpy, python et TensorFlow

F = make_features(load_prices())
n_train, origs = origins(len(F))
FEAT = ["ret_KO", "ret_PEP", "vol21_KO"]
Fv, ret, logKO = F[FEAT].values, F["ret_KO"].values, F["log_KO"].values
NF, I0 = len(FEAT), 21                       # I0 : 1re ligne sans NaN (la volatilité 21 j démarre à 21)
sd = np.nanstd(ret[1:n_train])               # échelle des rendements (calculée sur le TRAIN seulement)

## Fenêtres glissantes

In [ ]:
def make_windows(L, ends):
    """X : L jours jusqu'à t inclus ; Y : les H rendements suivants. Construction 100 % causale."""
    X = np.stack([Fv[t - L + 1:t + 1] for t in ends])
    Y = np.stack([ret[t + 1:t + 1 + H] for t in ends])
    return X, Y

def train_ends(L):
    return list(range(I0 + L - 1, n_train - 1 - H + 1))   # la cible reste dans le train (aucune fuite vers le test)

def build(kind, L, dropout):
    cell = layers.LSTM if kind == "LSTM" else layers.GRU
    m = keras.Sequential([layers.Input((L, NF)), cell(32, return_sequences=True), layers.Dropout(dropout),
                          cell(16), layers.Dropout(dropout), layers.Dense(H)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m

def fit(kind, L, dropout, Xtr, Ytr, Xva, Yva, epochs=40):
    """Normalisation ajustée sur le train SEUL ; arrêt précoce sur la validation (contre le surapprentissage)."""
    sc = StandardScaler().fit(Xtr.reshape(-1, NF))
    f = lambda X: sc.transform(X.reshape(-1, NF)).reshape(X.shape)
    m = build(kind, L, dropout)
    h = m.fit(f(Xtr), Ytr / sd, validation_data=(f(Xva), Yva / sd), epochs=epochs, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    return m, sc, h

def split_val(n, frac=0.2):
    """Validation = fin de la période d'apprentissage, séparée par un embargo de H jours (cibles chevauchantes)."""
    cut = int(n * (1 - frac))
    return slice(0, cut - H), slice(cut, n)

## Choix des hyperparamètres (grille réduite, sélection sur la perte de validation)

In [ ]:
# Petit réseau (2 couches, 32/16 unités) car ~1300 fenêtres seulement : un gros réseau surapprendrait.
# On compare 2 tailles de fenêtre (10 j ≈ 2 semaines, 30 j ≈ 1,5 mois) et 2 niveaux de dropout.
rows, models, hists = [], {}, {}
for kind in ["LSTM", "GRU"]:
    for L in [10, 30]:
        for dr in [0.1, 0.3]:
            X, Y = make_windows(L, train_ends(L))
            a, b = split_val(len(X))
            with Timer(f"{kind}_L{L}_d{dr}"):
                m, sc, h = fit(kind, L, dr, X[a], Y[a], X[b], Y[b])
            vl = min(h.history["val_loss"])
            rows.append({"modele": kind, "fenetre": L, "dropout": dr, "val_loss": vl, "epochs": len(h.history["loss"])})
            models[(kind, L, dr)], hists[(kind, L, dr)] = (m, sc), h
            print(f"  {kind} L={L} dropout={dr} -> val_loss={vl:.4f}")
hp = pd.DataFrame(rows)
hp.to_csv(RES_DIR / "dl_hyperparams.csv", index=False)
best_cfg = {k: tuple(hp[hp.modele == k].sort_values("val_loss").iloc[0][["fenetre", "dropout"]]) for k in ["LSTM", "GRU"]}
best_cfg = {k: (int(v[0]), float(v[1])) for k, v in best_cfg.items()}
print("\nMeilleures configurations :", best_cfg)

# Courbes de perte train/validation du modèle retenu (consigne : documenter l'entraînement).
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a_, kind in zip(ax, ["LSTM", "GRU"]):
    L, dr = best_cfg[kind]; h = hists[(kind, L, dr)]
    a_.plot(h.history["loss"], label="train"); a_.plot(h.history["val_loss"], label="validation")
    a_.set_title(f"{kind} (fenêtre {L}, dropout {dr}) — perte MSE"); a_.set_xlabel("epoch"); a_.legend()
savefig(fig, "fig15_dl_pertes.png")

## Validation croisée temporelle (walk-forward, fenêtre extensible) — jamais de k-fold classique

In [ ]:
# 5 plis : on entraîne sur le passé, on teste sur le bloc suivant (gap=H pour éviter le chevauchement).
# Métrique : RMSE sur le rendement cumulé à 10 jours, comparée à la baseline 'rendement nul' (= marche aléatoire).
cv_rows = []
for kind in ["LSTM", "GRU"]:
    L, dr = best_cfg[kind]
    X, Y = make_windows(L, train_ends(L))
    for k, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5, gap=H).split(X), 1):
        a, b = split_val(len(tr), 0.15)
        m, sc, _ = fit(kind, L, dr, X[tr][a], Y[tr][a], X[tr][b], Y[tr][b], epochs=30)
        pred = m.predict(sc.transform(X[te].reshape(-1, NF)).reshape(X[te].shape), verbose=0) * sd
        rmse = np.sqrt(np.mean((pred.sum(1) - Y[te].sum(1)) ** 2))
        rmse_naif = np.sqrt(np.mean(Y[te].sum(1) ** 2))
        cv_rows.append({"modele": kind, "pli": k, "RMSE_cum10j": rmse, "RMSE_naif": rmse_naif})
        print(f"  {kind} pli {k} : RMSE={rmse:.4f} | naïf={rmse_naif:.4f}")
cv = pd.DataFrame(cv_rows)
cv.to_csv(RES_DIR / "dl_cv_walkforward.csv", index=False)
print("\nMoyenne par modèle :\n", cv.groupby("modele")[["RMSE_cum10j", "RMSE_naif"]].agg(["mean", "std"]).round(4))
fig, ax = plt.subplots(figsize=(9, 4))
w = 0.25
for i, kind in enumerate(["LSTM", "GRU"]):
    s = cv[cv.modele == kind]; ax.bar(s.pli + (i - 0.5) * w, s.RMSE_cum10j, w, label=kind)
ax.plot(cv[cv.modele == "LSTM"].pli, cv[cv.modele == "LSTM"].RMSE_naif, "k--o", label="baseline naïve")
ax.set_xlabel("pli (walk-forward)"); ax.set_ylabel("RMSE rendement cumulé 10 j"); ax.legend(); ax.set_title("Validation croisée temporelle")
savefig(fig, "fig16_dl_cv_walkforward.png")
log_json("key_results.json", dl_best_cfg=str(best_cfg), dl_cv_mean=cv.groupby("modele")["RMSE_cum10j"].mean().to_dict(),
         dl_cv_naive_mean=float(cv["RMSE_naif"].mean()))

## Prévisions h=10 sur les mêmes blocs de test que les modèles économétriques

In [ ]:
# Les réseaux sont entraînés UNE fois (train) ; les modèles économétriques sont ré-estimés à chaque bloc.
# Cette asymétrie (coût de calcul) est signalée dans les limites du rapport.
for kind in ["LSTM", "GRU"]:
    L, dr = best_cfg[kind]
    m, sc = models[(kind, L, dr)]
    X = np.stack([Fv[o - L:o] for o in origs])                      # fenêtre qui finit en o-1 (dernier jour connu)
    pred = m.predict(sc.transform(X.reshape(-1, NF)).reshape(X.shape), verbose=0) * sd
    save_forecasts(kind, np.exp(logKO[np.array(origs) - 1][:, None] + np.cumsum(pred, axis=1)))
    print("  prévisions OK :", kind)

# 06 — Évaluation comparative et test de Diebold-Mariano (section 6)

In [ ]:
from common import *

px = load_prices()
n_train, origs = origins(len(px))
KO = px["KO"].values
actual = np.array([KO[o:o + H] for o in origs])             # (n_blocs, H) prix observés du test
ORDER = ["naive", "ARIMA_manuel", "ARIMA_auto", "SARIMA", "VAR", "VECM", "LSTM", "GRU"]
fc = {m: pd.read_csv(RES_DIR / f"fc_{m}.csv").values for m in ORDER if (RES_DIR / f"fc_{m}.csv").exists()}
print("Modèles évalués :", list(fc), f"| {len(origs)} blocs × {H} pas = {actual.size} prévisions chacun")

In [ ]:
# Tableau récapitulatif UNIQUE : mêmes observations, mêmes métriques pour tous les modèles.
COMPLEXITE = {"naive": "Aucune", "ARIMA_manuel": "Faible", "ARIMA_auto": "Faible", "SARIMA": "Moyenne", "VAR": "Moyenne",
              "VECM": "Moyenne", "LSTM": "Élevée", "GRU": "Élevée"}
INTERPRET = {"naive": "Totale", "ARIMA_manuel": "Élevée", "ARIMA_auto": "Élevée", "SARIMA": "Élevée", "VAR": "Élevée (Granger)",
             "VECM": "Élevée (équilibre LT)", "LSTM": "Faible (boîte noire)", "GRU": "Faible (boîte noire)"}
tim = read_json("timings.json")
lstm_t = {"LSTM": [v for k, v in tim.items() if k.startswith("LSTM_")], "GRU": [v for k, v in tim.items() if k.startswith("GRU_")]}
rows = []
for m, f in fc.items():
    t = tim.get(m) if m in tim else (np.mean(lstm_t[m]) if m in lstm_t and lstm_t[m] else np.nan)
    rows.append({"modele": m, **metrics(actual.ravel(), f.ravel()), "temps_fit_s": t,
                 "complexite": COMPLEXITE[m], "interpretabilite": INTERPRET[m]})
tab = pd.DataFrame(rows).sort_values("RMSE")
tab.to_csv(RES_DIR / "metrics.csv", index=False)
print("\n", tab.round(4).to_string(index=False))

In [ ]:
# Test de Diebold-Mariano : une différence de RMSE n'est utile que si elle est statistiquement significative.
err = {m: (actual - f).ravel() for m, f in fc.items()}
names = list(fc)
pmat = pd.DataFrame(np.nan, index=names, columns=names)
dm_rows = []
for a in names:
    for b in names:
        if a != b:
            pmat.loc[a, b] = dm_test(err[a], err[b])[1]
    if a != "naive":
        stat, p = dm_test(err["naive"], err[a])          # stat > 0 : le modèle 'a' bat la baseline
        dm_rows.append({"modele": a, "DM_stat_vs_naive": stat, "p_value": p,
                        "conclusion": ("meilleur que la baseline" if stat > 0 else "moins bon que la baseline") + (
                            " — SIGNIFICATIF (5 %)" if p < 0.05 else " — NON significatif")})
pmat.to_csv(RES_DIR / "dm_pvalues.csv")
dmv = pd.DataFrame(dm_rows)
dmv.to_csv(RES_DIR / "dm_vs_naive.csv", index=False)
print("\nDiebold-Mariano contre la baseline naïve (perte quadratique) :\n", dmv.round(4).to_string(index=False))

In [ ]:
# Conclusion automatique, à reformuler dans le rapport en tenant compte de l'interprétabilité et du coût.
best = tab.iloc[0]["modele"]
print(f"\nMeilleur RMSE : {best} ({tab.iloc[0]['RMSE']:.4f} USD) ; baseline naïve : {tab.loc[tab.modele == 'naive', 'RMSE'].iloc[0]:.4f} USD")
sig = dmv[dmv.p_value < 0.05]
print("Modèles significativement différents de la baseline :", list(sig.modele) or "AUCUN (cas fréquent sur actions liquides : marché quasi efficient)")
log_json("key_results.json", best_rmse_model=best, models_sig_vs_naive=list(sig.modele))

In [ ]:
# Figures : prévisions recollées sur tout le test, barres de RMSE, carte des p-values DM.
dates_blocks = np.concatenate([px.index[o:o + H] for o in origs])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates_blocks, actual.ravel(), "k", lw=1.8, label="observé")
for m, f in fc.items():
    for k in range(len(origs)):
        ax.plot(px.index[origs[k]:origs[k] + H], f[k], lw=0.9, color=plt.cm.tab10(ORDER.index(m)), label=m if k == 0 else None)
ax.legend(ncol=4, fontsize=8); ax.set_title(f"Prévisions à {H} jours sur l'échantillon test (blocs de {H} jours, KO en USD)")
savefig(fig, "fig17_previsions_test.png")

fig, ax = plt.subplots(figsize=(9, 4))
t = tab.sort_values("RMSE")
ax.barh(t.modele, t.RMSE, color="steelblue"); ax.invert_yaxis(); ax.set_xlabel("RMSE (USD)"); ax.set_title("RMSE par modèle (plus petit = meilleur)")
savefig(fig, "fig18_metriques.png")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(pmat.values.astype(float), vmin=0, vmax=0.2, cmap="RdYlGn_r")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        if i != j:
            ax.text(j, i, f"{pmat.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="p-value DM (rouge < 0.05 : différence significative)"); ax.set_title("Test de Diebold-Mariano — p-values par paires")
savefig(fig, "fig19_dm_pvalues.png")

# 07 — Section 5 (IA générative) : génération des prompts à partir de VOS résultats
Ce script ne parle pas à un LLM : il prépare les prompts. Vous les collez dans ChatGPT / Claude / Gemini,
puis vous collez la réponse dans llm/reponses/ (ou directement dans l'application Streamlit).

In [ ]:
from common import *

key = read_json("key_results.json")
def csv_txt(name, nd=4):
    p = RES_DIR / name
    return pd.read_csv(p).round(nd).to_string(index=False) if p.exists() else "(fichier absent : exécuter les scripts précédents)"

desc = pd.read_csv(RES_DIR / "stats_descriptives.csv", index_col=0).round(5).to_string() if (RES_DIR / "stats_descriptives.csv").exists() else "(absent)"
CTX = "Contexte : séries journalières Coca-Cola (KO) et PepsiCo (PEP), 2019-2024 (Yahoo Finance, prix ajustés), split chronologique 90/10.\n"

In [ ]:
P1 = CTX + f"""
Tu es un expert en séries temporelles. Voici les statistiques descriptives des rendements log (échantillon d'apprentissage) :
{desc}

Résultats des tests de stationnarité (ADF : H0 = racine unitaire ; KPSS : H0 = stationnaire) :
{csv_txt('stationnarite.csv')}

TÂCHE : propose une liste ordonnée d'hypothèses de modèles à tester (ARIMA, SARIMA, VAR, cointégration/VECM, GARCH, réseaux
récurrents...). Pour chaque modèle, justifie par les chiffres ci-dessus et indique quel diagnostic le validerait ou l'invaliderait.
N'invente aucun chiffre : si une information manque, dis-le.
"""

P2 = CTX + f"""
Tu t'adresses à un décideur NON statisticien (directeur financier). Explique simplement, sans jargon, les résultats suivants.

Tableau de comparaison des modèles de prévision à 10 jours (prix de KO en USD ; RMSE/MAE en dollars, MAPE en %) :
{csv_txt('metrics.csv')}

Test de Diebold-Mariano contre la baseline naïve :
{csv_txt('dm_vs_naive.csv')}

Résultats clés : {json.dumps({k: key.get(k) for k in ['granger_PEP_to_KO_p','granger_KO_to_PEP_p','engle_granger_p','johansen_rank','garch_best','garch_persistence','garch_half_life_days','best_arima','best_rmse_model']}, ensure_ascii=False)}

TÂCHE : (1) ce que les résultats disent de KO et PEP, (2) ce que signifient cointégration, Granger et volatilité conditionnelle
en langage courant, (3) le modèle à retenir et pourquoi. N'invente aucune p-value ni aucun chiffre absent du texte ci-dessus.
"""

P3 = CTX + f"""
À partir des performances ci-dessous, formule des recommandations d'investissement SIMULÉES (exercice pédagogique) sur KO et PEP :
{csv_txt('metrics.csv')}
Volatilité prévue par le GARCH sur les 10 prochains jours (écart-type journalier en %) :
{csv_txt('garch_vol_forecast.csv')}

TÂCHE : (1) donne tes recommandations ; (2) TOI-MÊME, liste ensuite explicitement les limites, risques et hypothèses fragiles de ces
recommandations (efficience des marchés, coûts de transaction, instabilité des paramètres, risque de modèle, surapprentissage...).
"""

for name, txt in [("01_hypotheses_modeles", P1), ("02_vulgarisation", P2), ("03_recommandations", P3)]:
    (LLM_DIR / "prompts" / f"{name}.md").write_text(txt.strip() + "\n", encoding="utf-8")
    rep = LLM_DIR / "reponses" / f"{name}.md"
    if not rep.exists():
        rep.write_text("(Collez ici la réponse INTÉGRALE du LLM, sans la modifier.)\n", encoding="utf-8")

In [ ]:
# Modèles de fichiers à remplir par l'étudiant (la consigne exige que l'interprétation experte soit écrite AVANT de solliciter le LLM).
(LLM_DIR / "interpretation_humaine.md").write_text(
    "# Mon interprétation experte (à rédiger AVANT de lire la réponse du LLM)\n\n"
    "## Stationnarité et choix de d\n\n## Modèle ARIMA retenu et diagnostic des résidus\n\n"
    "## VAR, Granger, cointégration / VECM\n\n## Volatilité conditionnelle (GARCH)\n\n"
    "## LSTM/GRU vs modèles classiques\n\n## Meilleur modèle et pourquoi (performance, interprétabilité, coût)\n", encoding="utf-8")
(LLM_DIR / "comparaison_critique.md").write_text(
    "# Comparaison humain / LLM — détection des erreurs (hallucinations statistiques)\n\n"
    "| Affirmation du LLM | Vérification (mes résultats / script) | Verdict (correct / simplifié / faux / inventé) | Correction |\n"
    "|---|---|---|---|\n| | | | |\n\n"
    "## Où l'interprétation du LLM converge avec la mienne\n\n## Où le LLM se trompe, simplifie ou invente une causalité\n\n"
    "## Ce que le LLM m'a fait gagner (temps, clarté) et ce qu'il n'a pas su faire\n", encoding="utf-8")
print("Prompts générés dans llm/prompts/ — copiez-les dans un LLM, puis collez les réponses dans llm/reponses/.")